In [1]:
import json
import plotly.graph_objects as go
import numpy as np

def load_json(path):
    with open(path) as f:
        return json.load(f)

def plot_interpolation_metric(data, metric_key='train_accuracy', use_log_scale=False, title_suffix=""):
    """
    Plot mean and std dev of a metric across model pairs for aligned and unaligned interpolations.
    
    Parameters:
    -----------
    data : list
        List of dictionaries containing interpolation results
    metric_key : str, default='train_accuracy'
        Key to extract from interpolation results (e.g., 'train_accuracy', 'train_loss', 'val_accuracy')
    use_log_scale : bool, default=False
        Whether to use log scale for y-axis
    """
    # Extract metrics for both aligned and unaligned
    aligned_metrics = []
    unaligned_metrics = []
    lambdas = None
    
    for pair in data:
        aligned_metrics.append(pair['interpolation_output_aligned'][metric_key])
        unaligned_metrics.append(pair['interpolation_output_unaligned'][metric_key])
        if lambdas is None:
            lambdas = pair['interpolation_output_aligned']['lambdas']
    
    # Convert to numpy arrays
    aligned_metrics = np.array(aligned_metrics)
    unaligned_metrics = np.array(unaligned_metrics)
    
    # Calculate mean and std
    aligned_mean = np.mean(aligned_metrics, axis=0)
    aligned_min = np.min(aligned_metrics, axis=0)
    aligned_max = np.max(aligned_metrics, axis=0)
    aligned_std = np.std(aligned_metrics, axis=0)
    unaligned_mean = np.mean(unaligned_metrics, axis=0)
    unaligned_min = np.min(unaligned_metrics, axis=0)
    unaligned_max = np.max(unaligned_metrics, axis=0)
    unaligned_std = np.std(unaligned_metrics, axis=0)
    
    # Create the plot
    fig = go.Figure()
    
    # Add aligned traces
    fig.add_trace(go.Scatter(
        x=lambdas,
        y=aligned_max,
        mode='lines+markers',
        name='Aligned (Max)',
        line=dict(color='blue', width=0.3),
    ))
    fig.add_trace(go.Scatter(
        x=lambdas,
        y=aligned_mean,
        mode='lines+markers',
        name='Aligned (Mean)',
        line=dict(color='blue', width=2),
    ))
    fig.add_trace(go.Scatter(
        x=lambdas,
        y=aligned_min,
        mode='lines+markers',
        name='Aligned (Min)',
        line=dict(color='blue', width=0.3),
    ))
    
    fig.add_trace(go.Scatter(
        x=lambdas + lambdas[::-1],
        y=np.concatenate([aligned_mean + aligned_std, (aligned_mean - aligned_std)[::-1]]),
        fill='toself',
        fillcolor='rgba(0, 100, 255, 0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        showlegend=True,
        name='Aligned (±1 Std)',
    ))
    
    # Add unaligned traces
    fig.add_trace(go.Scatter(
        x=lambdas,
        y=unaligned_max,
        mode='lines+markers',
        name='Unaligned (Max)',
        line=dict(color='red', width=0.3),
    ))
    fig.add_trace(go.Scatter(
        x=lambdas,
        y=unaligned_mean,
        mode='lines+markers',
        name='Unaligned (Mean)',
        line=dict(color='red', width=2),
    ))
    fig.add_trace(go.Scatter(
        x=lambdas,
        y=unaligned_min,
        mode='lines+markers',
        name='Unaligned (Min)',
        line=dict(color='red', width=0.3),
    ))
    
    fig.add_trace(go.Scatter(
        x=lambdas + lambdas[::-1],
        y=np.concatenate([unaligned_mean + unaligned_std, (unaligned_mean - unaligned_std)[::-1]]),
        fill='toself',
        fillcolor='rgba(255, 0, 0, 0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        showlegend=True,
        name='Unaligned (±1 Std)',
    ))
    
    fig.update_layout(
        title=f'{metric_key.replace("_", " ").title()} Across Model Interpolation ({title_suffix})',
        xaxis_title='Lambda (Interpolation Parameter)',
        yaxis_title=metric_key.replace('_', ' ').title(),
        yaxis_type='log' if use_log_scale else 'linear',
        hovermode='x unified',
        template='plotly_white',
    )
    
    return fig

In [7]:
results_mlp = load_json("outputs/2025-11-18/16-43-22__mlp_mnist/rebasin_lmc_results_epoch_100.json")
plot_interpolation_metric(results_mlp, "val_loss", False, title_suffix="MLPs")

In [8]:
results_wmlp = load_json("outputs/2025-11-17/17-10-20_mlp_mnist_sym-1__8y7giw5t__zzae2z6u/rebasin_lmc_results_epoch_100.json")
plot_interpolation_metric(results_wmlp, "val_loss", False, title_suffix="W-MLPs with κ=0, treated as MLPs during alignment and inference")

In [15]:
results_cifar_wrong = load_json("outputs/2025-11-19/22-41-20__resnet_cifar/rebasin_lmc_results_epoch_100.json")
plot_interpolation_metric(results_cifar_wrong, "train_accuracy", True, title_suffix="MLPs")

In [28]:
results_cifar = load_json("outputs/2025-11-20/20-04-51_resnet_cifar_sym-0__136hb7xt__npaor8q2/rebasin_lmc_results_epoch_100.json")
plot_interpolation_metric(results_cifar, "train_accuracy", False, title_suffix="ResNets")

In [29]:
results_cifar = load_json("outputs/2025-11-20/23-19-11_resnet_cifar_sym-1__136hb7xt__7tkmwgyh/rebasin_lmc_results_epoch_100.json")
plot_interpolation_metric(results_cifar, "train_accuracy", False, title_suffix="W-ResNets with κ=0, treated as ResNets during alignment and inference")